In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# SCOPING REVIEW – MASTER FIGURE GENERATION SCRIPT (BMJ HCI PUBLICATION-READY)
# ═══════════════════════════════════════════════════════════════════════════════
# Output: true vector PDF (editable text, embedded Type 42 fonts) + 600 dpi PNG preview
#         saved into /content/figures (Colab) or ./figures (local)
# ═══════════════════════════════════════════════════════════════════════════════

import os
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['svg.fonttype'] = 'none'

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, FancyBboxPatch
import numpy as np

OUTPUT_DIR = "./figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']

BAR_MAIN   = '#2b5c8f'
BAR_PEAK   = '#1a365d'
BAR_PART   = '#7ca3cd'
BAR_ABSENT = '#94a3b8'
BG_AX      = '#f8fafc'
SPINE_COL  = '#cbd5e1'
GRID_COL   = '#e2e8f0'
INK        = '#111111'
T_GREY     = '#64748b'

CONCORD_BORDER = '#1a365d'
CONCORD_FILL   = '#ffffff'
DISCORD_FILL   = '#fbe4e4'
DISCORD_TEXT   = '#b91c1c'
UNVER_FILL     = '#eef3fb'
UNVER_BORDER   = '#a9bdd6'

FS_AX      = 10.0
FS_VAL     = 10.0
FS_LAB     = 11.0
FS_PAN     = 12.0
FS_SUP     = 14.0

def _panel_title(ax, letter, title_text):
    ax.set_title(f"{letter}   {title_text}", loc='left', fontsize=FS_PAN,
                 fontweight='bold', pad=48, color=INK)

def _suptitle(fig, text, y_pos=0.96):
    fig.suptitle(text, fontsize=FS_SUP, fontweight='bold', color=INK, y=y_pos)

def _save(fig, filename):
    pdf_path = os.path.join(OUTPUT_DIR, f"{filename}.pdf")
    png_path = os.path.join(OUTPUT_DIR, f"{filename}.png")
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(png_path, format='png', dpi=600, bbox_inches='tight', facecolor='white')
    print(f"   ✅ Saved: {pdf_path}")
    print(f"   ✅ Saved: {png_path}")
    plt.close(fig)

def _horiz_bar(ax, categories, values, letter, title_text,
               colors, xpad=2.0, val_fmt=None):
    ax.set_facecolor(BG_AX)
    y_pos = np.arange(len(categories))
    hbars = ax.barh(y_pos, values, color=colors, height=0.58, zorder=3, edgecolor='white', linewidth=0.5)
    for bar, v in zip(hbars, values):
        w = bar.get_width()
        label = val_fmt(v) if val_fmt else f'n = {int(w)}'
        ax.annotate(label, xy=(w, bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords='offset points', va='center', ha='left',
                    fontsize=FS_VAL, fontweight='bold', color=INK)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(categories, fontsize=FS_AX, color=INK)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Studies', fontsize=FS_LAB, color=INK, labelpad=5)
    ax.set_xlim(0, max(values) + xpad)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(SPINE_COL); ax.spines['bottom'].set_color(SPINE_COL)
    ax.grid(axis='x', linestyle=':', alpha=0.55, color=GRID_COL)
    ax.grid(axis='y', visible=False)
    ax.tick_params(axis='y', length=0, pad=5)
    ax.tick_params(axis='x', labelsize=9.5)
    _panel_title(ax, letter, title_text)


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — PRISMA 2020 FLOW DIAGRAM
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure1_prisma():
    print("⏳ Generating Figure 1 (PRISMA Flow Diagram)...")
    fig, ax = plt.subplots(figsize=(19.0, 15.0))
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 110)
    ax.axis('off')

    BAND_LIGHT = '#f1f5f9'
    BAND_WHITE = '#ffffff'

    def band(y0, y1, color, label):
        ax.add_patch(Rectangle((0, y0), 200, y1 - y0, facecolor=color, edgecolor='none', zorder=0))
        if label:
            ax.text(1.5, y1 - 2.5, label, fontsize=12.0, fontweight='bold', color='#334155', va='top', ha='left', zorder=6, linespacing=1.35)

    band(0, 20, BAND_LIGHT, 'INCLUDED\n(n = 21)')
    band(20, 78, BAND_WHITE, 'SCREENING')
    band(78, 110, BAND_LIGHT, 'IDENTIFICATION')
    for y in [20, 78]:
        ax.plot([0, 200], [y, y], color='#cbd5e1', lw=1.2, zorder=1)

    def add_box(xc, yc, w, h, text, fontsize=11.0, facecolor='white',
                edgecolor='#334155', weight='normal', lw=1.1, zorder=3, ha='center'):
        ax.add_patch(Rectangle((xc - w / 2, yc - h / 2), w, h,
                                facecolor=facecolor, edgecolor=edgecolor,
                                linewidth=lw, zorder=zorder))
        tx = (xc - w / 2 + 2.5) if ha == 'left' else xc
        ax.text(tx, yc, text, fontsize=fontsize, color=INK, ha=ha,
                 va='center', linespacing=1.35, fontweight=weight, zorder=zorder + 1)

    def add_arrow(x1, y1, x2, y2, lw=1.3):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                     arrowprops=dict(arrowstyle='-|>', color='#334155', lw=lw,
                                     shrinkA=0, shrinkB=0, mutation_scale=15),
                     zorder=2)

    def add_header(x0, x1, yc, h, text):
        ax.add_patch(FancyBboxPatch((x0, yc - h / 2), x1 - x0, h,
                                     boxstyle="round,pad=0.35,rounding_size=0.5",
                                     facecolor='#dbe5f1', edgecolor='#94b0d1',
                                     linewidth=1.1, zorder=3))
        ax.text((x0 + x1) / 2, yc, text, fontsize=13.0, fontweight='bold', color=INK,
                 ha='center', va='center', zorder=4)

    # Core meticulously balanced coordinates
    C1, W1 = 28, 38
    EX1, W_EX1 = 78, 46
    C3, W3 = 128, 38
    EX2, W_EX2 = 176, 44

    add_header(7, 102, 99.0, 5.0, 'Databases & Registers')
    add_header(107, 199, 99.0, 5.0, 'Citation Tracking')

    yA = 89
    add_box(C1, yA, W1, 10, 'Records identified from:\nPubMed, MEDLINE, Embase, Scopus\n(n = 1,234)', fontsize=10.5)
    add_box(EX1, yA, W_EX1, 10, 'Records removed before screening:\nDuplicate records removed\n(n = 604)', fontsize=10.0)
    add_box(C3, yA, W3, 10, 'Records identified via citation\ntracking & manual search\n(n = 808)', fontsize=10.5)
    add_arrow(C1 + W1/2, yA, EX1 - W_EX1/2, yA)

    yB = 68
    add_box(C1, yB, W1, 10, 'Records screened\n(n = 630)', fontsize=11.0)
    add_box(EX1, yB, W_EX1, 10, 'Records excluded at title\nand abstract stage\n(n = 507)', fontsize=10.0)
    add_box(C3, yB, W3, 10, 'Leads screened\n(n = 808)', fontsize=11.0)
    add_box(EX2, yB, W_EX2, 10, 'Leads excluded at title\nand abstract stage\n(n = 632)', fontsize=10.0)

    add_arrow(C1, yA - 5, C1, yB + 5)
    add_arrow(C1 + W1/2, yB, EX1 - W_EX1/2, yB)
    add_arrow(C3, yA - 5, C3, yB + 5)
    add_arrow(C3 + W3/2, yB, EX2 - W_EX2/2, yB)

    yC = 49
    add_box(C1, yC, W1, 10, 'Reports sought for retrieval\n(n = 123)', fontsize=10.5)
    add_box(C3, yC, W3, 10, 'Reports sought for retrieval\n(n = 176)', fontsize=10.5)
    add_arrow(C1, yB - 5, C1, yC + 5)
    add_arrow(C3, yB - 5, C3, yC + 5)

    yE = 33
    add_box(C1, yE, W1, 8, 'Reports assessed (n = 123)', fontsize=10.5)
    add_box(C3, yE, W3, 8, 'Reports assessed (n = 176)', fontsize=10.5)
    add_arrow(C1, yC - 5, C1, yE + 4)
    add_arrow(C3, yC - 5, C3, yE + 4)

    yD = 33
    box_h = 24
    add_box(EX1, yD, W_EX1, box_h,
            'Reports excluded (n = 106):\n'
            '• Secondary reviews routed to citation tracking (n = 45)\n'
            '• Primary research excluded (n = 61):\n'
            '  - No human evaluation (n = 18)\n'
            '  - Conference abstracts (n = 15)\n'
            '  - Educational conversational models (n = 7)\n'
            '  - Full-text unavailable (n = 6)\n'
            '  - Clinician-facing XAI (n = 5)\n'
            '  - Trial registries (n = 4)\n'
            '  - Non-English (n = 2)\n'
            '  - Text simplification (n = 2)\n'
            '  - Attitude surveys (n = 1)\n'
            '  - Black-box risk calculator (n = 1)', fontsize=8.5, ha='left')

    add_box(EX2, yD, W_EX2, box_h,
            'Reports excluded (n = 172):\n'
            '• Black-box wearables (n = 44)\n'
            '• Black-box diagnostics (n = 31)\n'
            '• Clinician-facing XAI (n = 26)\n'
            '• Educational models (n = 25)\n'
            '• Attitude surveys (n = 16)\n'
            '• Text simplification (n = 14)\n'
            '• Risk calculators / Other (n = 16)', fontsize=9.5, ha='left')

    add_arrow(C1 + W1/2, yE, EX1 - W_EX1/2, yD, lw=1.4)
    add_arrow(C3 + W3/2, yE, EX2 - W_EX2/2, yD, lw=1.4)

    yF = 10
    add_box((C1+C3)/2, yF, 120, 12,
            'Studies included in review (n = 21)\n'
            '• Included via database searches (n = 17)\n'
            '• Included via citation tracking (n = 4)',
            fontsize=11.5, weight='bold', facecolor='#eef2f7', edgecolor='#94b0d1')

    add_arrow(C1, yE - 4, C1, yF + 6)
    add_arrow(C3, yE - 4, C3, yF + 6)

    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
    _save(fig, 'Figure_1_PRISMA_Flow_Diagram')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Clinical, Methodological and Technical Landscape
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure2_multipanel():
    print("⏳ Generating Figure 2 (Multi-Panel Bar)...")
    fig, axes = plt.subplots(1, 5, figsize=(23.0, 5.5), sharey=False)
    domains = ['Internal\nMedicine', 'Oncology', 'Primary /\nEmergency Care', 'Psychiatry', 'Surgery', 'Radiology']
    domain_counts = [5, 4, 4, 4, 2, 2]
    tasks = ['Diagnosis', 'Triage', 'Prognosis', 'Self-\nManagement', 'Screening', 'Treatment\nSelection']
    task_counts = [5, 4, 4, 4, 3, 1]
    designs = ['Qualitative', 'Randomised\nExperiment', 'Observational\nStudy', 'Randomised\nControlled Trial', 'Mixed Methods']
    design_counts = [6, 6, 5, 2, 2]
    architects = ['Simulated\nAlgorithm', 'Black-Box +\nPost-Hoc', 'LLM', 'Interpretable\nML']
    architect_counts = [7, 5, 5, 4]
    phases = ['Summative\nEvaluation', 'Needs\nAssessment', 'Co-design', 'Clinical\nImplementation']
    phase_counts = [12, 4, 3, 2]

    _horiz_bar(axes[0], domains, domain_counts, 'A', 'Clinical Domain', colors=[BAR_MAIN]*len(domains), xpad=2.0)
    _horiz_bar(axes[1], tasks, task_counts, 'B', 'Clinical Task', colors=[BAR_MAIN]*len(tasks), xpad=2.0)
    _horiz_bar(axes[2], designs, design_counts, 'C', 'Study Design', colors=[BAR_MAIN]*len(designs), xpad=2.0)
    _horiz_bar(axes[3], architects, architect_counts, 'D', 'AI Architecture', colors=[BAR_MAIN]*len(architects), xpad=2.0)
    _horiz_bar(axes[4], phases, phase_counts, 'E', 'Involvement Phase', colors=[BAR_MAIN]*len(phases), xpad=3.5)

    fig.subplots_adjust(left=0.03, right=0.99, top=0.65, bottom=0.15, wspace=0.38)
    _suptitle(fig, 'Clinical, Methodological and Technical Landscape', y_pos=0.92)
    _save(fig, 'Figure_2_Clinical_Methodological_Technical_Landscape')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Study Characteristics Table
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure3_table():
    print("⏳ Generating Figure 3 (Characteristics Table)...")
    columns = ['Author (Year)', 'Country', 'Clinical Domain', 'Clinical Task', 'Study Design',
               'Testing (n)', 'AI Architecture', 'Explanation Modality',
               'Delivery Format', 'Patient Involvement Phase']
    data = [
        ['Alam & Mueller (2021)', 'USA', 'Internal Medicine', 'Diagnosis', 'Randomised Experiment', '193', 'Simulated Algorithm', 'Hybrid', 'Web Platform', 'Summative Testing'],
        ['Alduhailan et al. (2025)', 'Saudi Arabia', 'Primary & Emergency Care', 'Triage', 'Observational Study', '35', 'Black-Box ML', 'Textual', 'Mobile Health App', 'Summative Testing'],
        ['Algarni (2025)', 'Saudi Arabia', 'Radiology', 'Diagnosis', 'Observational Study', 'Not Reported', 'Large Language Model', 'Hybrid', 'Conceptual Prototype', 'Summative Testing'],
        ['Aziz et al. (2026)', 'Pakistan', 'Psychiatry', 'Self-Management', 'Randomised Controlled Trial', '120', 'Large Language Model', 'Textual', 'Web Platform', 'Clinical Implementation'],
        ['Begum et al. (2025)', 'Pakistan', 'Internal Medicine', 'Self-Management', 'Observational Study', '200 (20)*', 'Black-Box ML', 'Textual', 'Mobile Health App', 'Clinical Implementation'],
        ['Bolpagni et al. (2024)', 'Italy', 'Psychiatry', 'Self-Management', 'Qualitative Study', '12', 'Simulated Algorithm', 'Textual', 'Conceptual Prototype', 'Co-design'],
        ['Gombolay et al. (2024)', 'USA', 'Internal Medicine', 'Treatment Choice', 'Randomised Experiment', '365', 'Black-Box ML', 'Visual', 'Web Platform', 'Summative Testing'],
        ['Gould et al. (2023)', 'Australia', 'Surgery', 'Prognosis', 'Qualitative Study', '20', 'Simulated Algorithm', 'Textual', 'Conceptual Prototype', 'Needs Assessment'],
        ['He et al. (2023)', 'China', 'Internal Medicine', 'Screening', 'Mixed Methods', '8', 'Simulated Algorithm', 'Hybrid', 'Conceptual Prototype', 'Co-design'],
        ['Hildebrand et al. (2023)', 'USA', 'Oncology', 'Prognosis', 'Qualitative Study', '30', 'Black-Box ML', 'Textual', 'Static Report', 'Summative Testing'],
        ['Huang et al. (2024)', 'USA', 'Primary & Emergency Care', 'Self-Management', 'Randomised Experiment', '155', 'Large Language Model', 'Textual', 'Web Platform', 'Summative Testing'],
        ['Lombaers et al. (2026)', 'Netherlands', 'Oncology', 'Prognosis', 'Qualitative Study', '19', 'Interpretable Machine Learning', 'Hybrid', 'Static Report', 'Needs Assessment'],
        ['Maywald et al. (2025)', 'Germany', 'Oncology', 'Self-Management', 'Observational Study', '231', 'Large Language Model', 'Hybrid', 'In-Clinic App', 'Summative Testing'],
        ['Metta et al. (2024)', 'Italy & France', 'Oncology', 'Diagnosis', 'Observational Study', '156', 'Black-Box ML', 'Hybrid', 'Web Platform', 'Summative Testing'],
        ['Sperling et al. (2025)', 'USA', 'Internal Medicine', 'Prognosis', 'Qualitative Study', '52', 'Simulated Algorithm', 'Textual', 'Conceptual Prototype', 'Needs Assessment'],
        ['Tsai et al. (2021)', 'USA', 'Primary & Emergency Care', 'Triage', 'Mixed Methods', '45', 'Interpretable Machine Learning', 'Textual', 'Web Platform', 'Needs Assessment'],
        ['Wang et al. (2025)', 'China', 'Psychiatry', 'Screening', 'Randomised Experiment', '346', 'Simulated Algorithm', 'Textual', 'Web Platform', 'Summative Testing'],
        ['Williams et al. (2025)', 'USA', 'Psychiatry', 'Screening', 'Qualitative Study', '33', 'Interpretable Machine Learning', 'Hybrid', 'Conceptual Prototype', 'Co-design'],
        ['Woodcock et al. (2021)', 'United Kingdom', 'Primary & Emergency Care', 'Triage', 'Randomised Experiment', '750', 'Simulated Algorithm', 'Textual', 'Web Platform', 'Summative Testing'],
        ['Wu et al. (2025)', 'China', 'Surgery', 'Screening', 'Randomised Controlled Trial', '70', 'Large Language Model', 'Hybrid', 'In-Clinic App', 'Summative Testing'],
        ['Zhang et al. (2021)', 'USA', 'Radiology', 'Self-Management', 'Randomised Experiment', '3,423', 'Interpretable Machine Learning', 'Textual', 'Web Platform', 'Summative Testing'],
    ]

    # ---------------------------------------------------------
    # UPDATED COLUMN WIDTHS:
    # Patient Phase down from 0.17 -> 0.13
    # AI Architecture up from 0.11 -> 0.13
    # Explanation Modality up from 0.08 -> 0.10
    # Overall sum remains 1.0
    # ---------------------------------------------------------
    col_widths = [0.13, 0.07, 0.11, 0.09, 0.10, 0.05, 0.13, 0.10, 0.09, 0.13]

    fig, ax = plt.subplots(figsize=(20.0, 9.2))
    ax.axis('off')
    tbl = ax.table(cellText=data, colLabels=columns, cellLoc='left', colWidths=col_widths, loc='upper center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 1.85)

    HEADER_BG = '#dbe5f1'
    ROW_ALT   = '#f0f4f9'

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor('#cbd5e1')
        cell.set_linewidth(0.6)
        cell.PAD = 0.03
        if r == 0:
            cell.set_facecolor(HEADER_BG)
            cell.set_text_props(fontweight='bold', color=INK, ha='left', va='center', fontsize=10.0)
        else:
            cell.set_facecolor(ROW_ALT if r % 2 == 0 else 'white')
            cell.set_text_props(color=INK, ha='left', va='center', fontsize=9.5)

    ax.text(0.0, -0.035, '* Testing N reflects longitudinal cohort (N=200); sub-sample (n=20) completed usability testing.',
            transform=ax.transAxes, fontsize=8.8, color=T_GREY, style='italic', ha='left', va='top')

    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.08)
    _suptitle(fig, 'Table 1 | Characteristics of included studies', y_pos=0.97)
    _save(fig, 'Figure_3_Study_Characteristics_Table')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Explanation Modality, Scope, & Delivery Format
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure4_explanation_delivery():
    print("⏳ Generating Figure 4 (Explanation Modalities)...")
    modalities = ['Textual', 'Hybrid', 'Visual']; modal_n = [12, 8, 1]
    scopes = ['Local', 'Local & Global', 'Global']; scope_n = [15, 5, 1]
    formats = ['Web Platform', 'Conceptual\nPrototype', 'Mobile\nHealth App', 'Static Report', 'In-Clinic App']; format_n = [9, 6, 2, 2, 2]
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 5.6), gridspec_kw={'wspace': 0.38})
    _horiz_bar(axes[0], modalities, modal_n, 'A', 'Explanation Modality', colors=[BAR_MAIN]*3, xpad=4.5)
    _horiz_bar(axes[1], scopes, scope_n, 'B', 'Explanation Scope', colors=[BAR_MAIN]*3, xpad=4.5)
    _horiz_bar(axes[2], formats, format_n, 'C', 'Delivery Format', colors=[BAR_MAIN]*5, xpad=4.5)
    fig.subplots_adjust(left=0.06, right=0.98, top=0.68, bottom=0.15, wspace=0.38)
    _suptitle(fig, 'Explanation Modality, Scope, and Delivery Format', y_pos=0.92)
    _save(fig, 'Figure_4_Explanation_Delivery')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — Patient Involvement Phase
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure5_patient_involvement():
    print("⏳ Generating Figure 5 (Patient Involvement)...")
    phases = ['Summative Evaluation', 'Needs Assessment', 'Co-design', 'Clinical Implementation']
    counts = [12, 4, 3, 2]; pcts = [c / 21 * 100 for c in counts]
    fig, ax = plt.subplots(figsize=(14.0, 5.4))
    ax.set_facecolor(BG_AX)
    y = np.arange(len(phases))
    bars = ax.barh(y, counts, color=BAR_MAIN, height=0.62, zorder=3, edgecolor='white', linewidth=0.5)
    for bar, v, p in zip(bars, counts, pcts):
        w = bar.get_width()
        ax.annotate(f'n = {v}   ({p:.0f}%)', xy=(w, bar.get_y() + bar.get_height() / 2), xytext=(8, 0), textcoords='offset points', va='center', ha='left', fontsize=FS_VAL, fontweight='bold', color=INK)
    ax.set_yticks(y); ax.set_yticklabels(phases, fontsize=FS_AX, color=INK); ax.invert_yaxis()
    ax.set_xlabel('Number of Studies', fontsize=FS_LAB, color=INK, labelpad=5)
    ax.set_xlim(0, max(counts) + 2.5)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(SPINE_COL); ax.spines['bottom'].set_color(SPINE_COL)
    ax.grid(axis='x', linestyle=':', alpha=0.55, color=GRID_COL)
    fig.subplots_adjust(left=0.14, right=0.98, top=0.78, bottom=0.15)
    _suptitle(fig, 'Patient Involvement Phase Across Included Studies', y_pos=0.93)
    _save(fig, 'Figure_5_Patient_Involvement')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 6 — Evaluation & Outcome Metrics
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure6_evaluation_metrics():
    print("⏳ Generating Figure 6 (Evaluation Metrics)...")
    subj_labels = ['Validated Psychometric Scales', 'Interviews & Focus Groups', 'Unvalidated Questionnaires']; subj_n = [11, 6, 4]
    obj_labels = ['Not Evaluated', 'Patient Comprehension & Accuracy', 'AI Algorithm Performance']; obj_n = [11, 5, 5]
    beh_labels = ['Not Evaluated', 'Experimental Setting', 'Real-World Clinical']; beh_n = [13, 6, 2]
    fig, axes = plt.subplots(3, 1, figsize=(14.0, 11.5), gridspec_kw={'hspace': 0.85})
    _horiz_bar(axes[0], subj_labels, subj_n, 'A', 'Subjective Metrics', colors=[BAR_MAIN]*3, xpad=1.5)
    _horiz_bar(axes[1], obj_labels, obj_n, 'B', 'Objective Metrics', colors=[BAR_ABSENT, BAR_MAIN, BAR_MAIN], xpad=1.5)
    _horiz_bar(axes[2], beh_labels, beh_n, 'C', 'Behavioural Metrics', colors=[BAR_ABSENT, BAR_MAIN, BAR_MAIN], xpad=1.5)
    ne = mpatches.Patch(facecolor=BAR_ABSENT, edgecolor=SPINE_COL, linewidth=0.7, label='Studies not reporting data in this domain')
    fig.subplots_adjust(left=0.16, right=0.98, top=0.88, bottom=0.10)
    fig.legend(handles=[ne], loc='lower center', bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=11)
    _suptitle(fig, 'Evaluation and Outcome Metrics', y_pos=0.96)
    _save(fig, 'Figure_6_Evaluation_Metrics')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 7 — Trust Calibration Matrix
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure7_trust_calibration_matrix():
    print("⏳ Generating Figure 7...")

    cells = {
        ('HIGH', 'accurate'): dict(
            title='Appropriate Reliance', n=5, pct=24, kind='concordant',
            studies=['Algarni (2025)', 'Aziz et al. (2026)', 'Begum et al. (2025)',
                     'Tsai (2021)', 'Wu et al. (2025)']),
        ('HIGH', 'inaccurate'): dict(
            title='Over-Reliance', n=4, pct=19, kind='discordant',
            studies=['Alduhailan et al. (2025)', 'Gombolay et al. (2024)',
                     'Maywald et al. (2025)', 'Metta et al. (2024)']),
        ('HIGH', 'unknown'): dict(
            title='Unverified Reliance', n=9, pct=43, kind='unverified',
            studies=['Alam & Mueller (2021)', 'Bolpagni et al. (2024)', 'He et al. (2023)',
                     'Hildebrand et al. (2023)', 'Huang et al. (2024)', 'Lombaers et al. (2026)',
                     'Wang et al. (2025)', 'Williams et al. (2025)', 'Woodcock et al. (2021)']),
        ('LOW', 'accurate'): dict(
            title='Under-Reliance', n=1, pct=5, kind='discordant',
            studies=['Gould et al. (2023)']),
        ('LOW', 'inaccurate'): dict(
            title='Appropriate Scepticism', n=1, pct=5, kind='concordant',
            studies=['Zhang et al. (2021)']),
        ('LOW', 'unknown'): dict(
            title='Unverified Scepticism', n=1, pct=5, kind='unverified',
            studies=['Sperling et al. (2025)']),
    }

    style_map = {
        'concordant': dict(facecolor=CONCORD_FILL, edgecolor=CONCORD_BORDER, lw=2.2, textcolor=INK),
        'discordant': dict(facecolor=DISCORD_FILL, edgecolor='#e2b4b4', lw=1.0, textcolor=DISCORD_TEXT),
        'unverified': dict(facecolor=UNVER_FILL, edgecolor=UNVER_BORDER, lw=1.0, textcolor=INK),
    }

    col_keys = ['accurate', 'inaccurate', 'unknown']
    col_titles = ['AI Model Accurate', 'AI Model Inaccurate', 'AI Model Unknown']
    row_keys = ['HIGH', 'LOW']
    row_titles = ['HIGH\nPatient Trust', 'LOW\nPatient Trust']

    fig, ax = plt.subplots(figsize=(14.0, 8.5))
    ax.set_xlim(0, 115)
    ax.set_ylim(0, 100)
    ax.axis('off')

    LABEL_C, LABEL_W = 8, 14
    COL_X = {'accurate': 32, 'inaccurate': 65, 'unknown': 98}
    COL_W = 28
    HEADER_Y, HEADER_H = 88, 9
    ROW_Y = {'HIGH': 62, 'LOW': 22}
    ROW_H = 32

    for k, title in zip(col_keys, col_titles):
        ax.add_patch(Rectangle((COL_X[k] - COL_W / 2, HEADER_Y - HEADER_H / 2), COL_W, HEADER_H,
                                facecolor='#eef2f7', edgecolor='#c9d3de', lw=0.8, zorder=2))
        ax.text(COL_X[k], HEADER_Y, title, fontsize=11, fontweight='bold', ha='center', va='center', color=INK, zorder=3)

    for rk, rtitle in zip(row_keys, row_titles):
        ax.add_patch(Rectangle((LABEL_C - LABEL_W / 2, ROW_Y[rk] - ROW_H / 2), LABEL_W, ROW_H,
                                facecolor='#eef2f7', edgecolor='#c9d3de', lw=0.8, zorder=2))
        ax.text(LABEL_C, ROW_Y[rk], rtitle, fontsize=10.5, fontweight='bold', ha='center', va='center', color=INK, zorder=3)

    for rk in row_keys:
        for ck in col_keys:
            d = cells[(rk, ck)]
            s = style_map[d['kind']]
            xc, yc = COL_X[ck], ROW_Y[rk]
            ax.add_patch(Rectangle((xc - COL_W / 2, yc - ROW_H / 2), COL_W, ROW_H,
                                    facecolor=s['facecolor'], edgecolor=s['edgecolor'],
                                    linewidth=s['lw'], zorder=2))
            top_y = yc + ROW_H / 2 - 3.2
            ax.text(xc - COL_W / 2 + 1.2, top_y, d['title'], fontsize=10, fontweight='bold',
                    color=INK, ha='left', va='top', zorder=3)
            ax.text(xc - COL_W / 2 + 1.2, top_y - 4.2, f"n = {d['n']} ({d['pct']}%)", fontsize=9.5,
                    fontweight='bold', color=s['textcolor'], ha='left', va='top', zorder=3)
            rule_y = top_y - 6.4
            ax.plot([xc - COL_W / 2 + 1.2, xc + COL_W / 2 - 1.2], [rule_y, rule_y],
                    color=s['edgecolor'], lw=0.8, alpha=0.7, zorder=3)
            study_text = '\n'.join(d['studies'])
            ax.text(xc - COL_W / 2 + 1.2, rule_y - 1.8, study_text, fontsize=7.6,
                    color=T_GREY, ha='left', va='top', linespacing=1.6, zorder=3)

    legend_items = [
        ('Concordant', dict(facecolor=CONCORD_FILL, edgecolor=CONCORD_BORDER, lw=2.2)),
        ('Discordant', dict(facecolor=DISCORD_FILL, edgecolor='#e2b4b4', lw=1.0)),
        ('Unverified', dict(facecolor=UNVER_FILL, edgecolor=UNVER_BORDER, lw=1.0)),
    ]
    lx = 30
    ax.text(15, 98.5, 'Category:', fontsize=10.5, fontweight='bold', ha='left', va='center', color=INK)
    for label, sty in legend_items:
        ax.add_patch(Rectangle((lx, 97), 3, 3, **sty, zorder=3))
        ax.text(lx + 4.2, 98.5, label, fontsize=10, ha='left', va='center', color=INK)
        lx += 20

    ax.set_title('Trust Calibration Matrix', fontsize=FS_SUP, fontweight='bold', color=INK, pad=20, loc='center')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.05)
    _save(fig, 'Figure_7_Trust_Calibration_Matrix')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 8 — Publication Timeline & Geographic Distribution
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure8_timeline_geography_final():
    print("⏳ Generating Figure 8...")
    years    = [2021, 2022, 2023, 2024, 2025, 2026]
    n_papers = [4,    0,    3,    4,    8,    2   ]
    months   = [12,   12,   12,   12,   12,   1.5 ]
    rate     = [n / m for n, m in zip(n_papers, months)]

    countries = ['USA', 'China', 'Saudi Arabia', 'Pakistan',
                 'Italy', 'Italy & France', 'United Kingdom',
                 'Germany', 'Australia', 'Netherlands']
    geo_n     = [8, 3, 2, 2, 1, 1, 1, 1, 1, 1]

    # Adjusted figsize to be slightly taller (5.0 instead of 4.8) for title breathing room
    fig, (ax_t, ax_g) = plt.subplots(
        1, 2, figsize=(10.0, 5.0),
        gridspec_kw={'width_ratios': [1.0, 1.15], 'wspace': 0.38}
    )

    bar_colors = [BAR_PEAK if y == 2025 else BAR_PART if y == 2026 else BAR_MAIN for y in years]
    x = np.arange(len(years))
    vbars = ax_t.bar(x, rate, color=bar_colors, width=0.40, zorder=3, edgecolor='white', linewidth=0.6)
    vbars[-1].set_hatch('////')
    vbars[-1].set_edgecolor(SPINE_COL)

    for bar, n, r in zip(vbars, n_papers, rate):
        if n == 0:
            ax_t.text(bar.get_x() + bar.get_width() / 2, 0.03, '0', ha='center', va='bottom', fontsize=10, color=SPINE_COL)
            continue
        ax_t.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03, f'n = {n}', ha='center', va='bottom', fontsize=10.5, fontweight='bold', color=INK)
        ax_t.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.12, f'({r:.2f}/mo)', ha='center', va='bottom', fontsize=9, color=T_GREY, style='italic')

    ax_t.set_facecolor(BG_AX)
    ax_t.set_xticks(x)
    ax_t.set_xticklabels([str(y) for y in years], fontsize=FS_AX)
    ax_t.set_xlabel('Publication Year', fontsize=FS_LAB, color=INK, labelpad=5)
    ax_t.set_ylabel('Studies per Month', fontsize=FS_LAB, color=INK, labelpad=5)

    # Reduced max Y-limit so the bars look proportionally taller, matching the reference image
    ax_t.set_ylim(0, 1.95)
    ax_t.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    for spine in ['top', 'right', 'left']: ax_t.spines[spine].set_visible(False)
    ax_t.spines['bottom'].set_color(SPINE_COL)
    ax_t.grid(axis='y', linestyle=':', alpha=0.55, color=GRID_COL)
    _panel_title(ax_t, 'A', 'Publication Timeline')

    # Updated legend labels to strictly match your correct image
    leg = [mpatches.Patch(facecolor=BAR_MAIN, label='Full calendar year'),
           mpatches.Patch(facecolor=BAR_PEAK, label='Peak year (2025)'),
           mpatches.Patch(facecolor=BAR_PART, edgecolor=SPINE_COL, hatch='////', label='Partial year (Jan-Feb 2026)')]
    ax_t.legend(handles=leg, loc='upper left', frameon=True, fontsize=9)

    _horiz_bar(ax_g, countries, geo_n, 'B', 'Geographic Distribution', colors=[BAR_MAIN]*len(geo_n), xpad=2.5)

    # FIXED OVERLAPPING TITLES:
    # Lowered the 'top' margin to 0.75 so the panel titles don't crash into the suptitle
    fig.subplots_adjust(left=0.07, right=0.97, top=0.75, bottom=0.14)
    _suptitle(fig, 'Publication Timeline and Geographic Distribution', y_pos=0.96)

    _save(fig, 'Figure_8_Timeline_Geography_Final')

if __name__ == "__main__":
    make_figure1_prisma()
    make_figure2_multipanel()
    make_figure3_table()
    make_figure4_explanation_delivery()
    make_figure5_patient_involvement()
    make_figure6_evaluation_metrics()
    make_figure7_trust_calibration_matrix()
    make_figure8_timeline_geography_final()
    print("💎 All finalized vector PDF + 600dpi PNG figures regenerated successfully.")

⏳ Generating Figure 1 (PRISMA Flow Diagram)...
   ✅ Saved: ./figures/Figure_1_PRISMA_Flow_Diagram.pdf
   ✅ Saved: ./figures/Figure_1_PRISMA_Flow_Diagram.png
⏳ Generating Figure 2 (Multi-Panel Bar)...
   ✅ Saved: ./figures/Figure_2_Clinical_Methodological_Technical_Landscape.pdf
   ✅ Saved: ./figures/Figure_2_Clinical_Methodological_Technical_Landscape.png
⏳ Generating Figure 3 (Characteristics Table)...
   ✅ Saved: ./figures/Figure_3_Study_Characteristics_Table.pdf
   ✅ Saved: ./figures/Figure_3_Study_Characteristics_Table.png
⏳ Generating Figure 4 (Explanation Modalities)...
   ✅ Saved: ./figures/Figure_4_Explanation_Delivery.pdf
   ✅ Saved: ./figures/Figure_4_Explanation_Delivery.png
⏳ Generating Figure 5 (Patient Involvement)...
   ✅ Saved: ./figures/Figure_5_Patient_Involvement.pdf
   ✅ Saved: ./figures/Figure_5_Patient_Involvement.png
⏳ Generating Figure 6 (Evaluation Metrics)...
   ✅ Saved: ./figures/Figure_6_Evaluation_Metrics.pdf
   ✅ Saved: ./figures/Figure_6_Evaluation_Metric